# This is a simple example script for 2D x/y stitching using muvis-align and the multiview-stitcher package

In [ ]:
from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import vis_utils
import networkx as nx

from src.muvis_align.MVSRegistration import MVSRegistration
from src.muvis_align.image.util import get_sim_physical_size, show_image
from src.muvis_align.util import print_dict_simple

## Initialise muvis-align, initialise sims, and pre-process

In [ ]:
reg = MVSRegistration(operation='register', input_pattern='../data/S000/*.zarr', output_pattern='../../output/', ui='mpl')
sims = reg.init_sims()
reg_sims, reg_indices = reg.preprocess(sims)

for label, sim in zip(reg.file_labels, sims):
    print(label, si_utils.get_origin_from_sim(sim), get_sim_physical_size(sim))

## Initialise registration parameters

In [ ]:
register_params = {
	'pairing': 'orthogonal',
	'transform_type': 'affine',
	'method': 'sift',
	'gaussian_sigma': 2,
	'normalisation': True,
	'max_keypoints': 5000,
	'inlier_threshold_factor': 0.05,
	'max_trials': 1000,
	'ransac_iterations': 3,
	'n_parallel_pairwise_regs': 1,
}

## Perform pair registration (using multiview-stitcher)

In [ ]:
g_reg, msims, registered_sims, pairs = reg.register_pairs(sims, register_sims=reg_sims, register_params=register_params)
qualities = {key: value.item() for key, value in nx.get_edge_attributes(g_reg, 'quality').items()}

print('quality')
print_dict_simple(qualities)

## Perfrom global registration (using multiview-stitcher)

In [ ]:
%matplotlib inline
results = reg.register_global(sims, msims, register_indices=reg_indices, register_params=register_params)

## Show registration mapping

In [ ]:
mappings = results['mappings']
for key, mapping in mappings.items():
    print(f'{reg.file_labels[key]}:\n', mapping.sel(t=0).data)

## Visualise registered sims

In [ ]:
%matplotlib inline
fig, ax = vis_utils.plot_positions(registered_sims, transform_key=reg.reg_transform_key, use_positional_colors=False, view_labels=reg.file_labels)

## Perform fusion (using multiview-stitcher)

In [ ]:
%matplotlib inline
fused, _ = reg.fuse(registered_sims)

fused

## Output fused result

In [ ]:
%matplotlib inline
show_image(fused[0, 0])

In [ ]:
reg.save(reg.output + 'stitching2d', fused)